# PerfectPetClub — Task 2: ETL Pipeline for Customer Churn
**Module:** 7COM2005 Data Engineering | **Team:** P4G5 | **Notebook:** `notebooks/etl_pipeline.ipynb`

**Goal:** build a clean, one-row-per-customer dataset for analysing **churn** (a customer cancelling the auto-renew of their subscription) and write it to Parquet for the PySpark ML notebook (Task 3).

### Pipeline architecture
```
EXTRACT                         TRANSFORM                                   LOAD
SQLite (9 tables)  ─┐           clean_* functions  (types, DQ checks)
CRM report (CSV)   ─┤──►  raw   ─► build_features()  (feature engineering) ─► validate ─► data/processed/
Server log (CSV)   ─┤           ─► leakage audit + final validation                         processed_churn.parquet
Survey (CSV)       ─┘
```

### Key design decisions (be ready to justify these)
1. **Target:** `churn = 1 - auto_renew` (the brief defines churn as cancelling auto-renew). 
2. **Data-leakage prevention:** `subscription.end_date`, tenure, "days since last order" and raw activity *counts* are **deliberately excluded**. Customers who cancel stop ordering and logging in, so these columns would reveal the label (a model would look brilliant but be useless). All behavioural features are **rates per month of each customer's own observation window**, or ratios/trends, so they do not encode how long someone stayed.
3. **No PII in the output:** name, e-mail, password hash and address are dropped before loading.
4. **Missing delivery dates are flagged, not invented:** 1,090 orders (4.7%) have no expected/delivered date, spread across the whole year (so not only recent orders). We cannot tell whether they are in transit or simply unrecorded, so we create a `delivery_missing` flag instead of fabricating dates.

In [ ]:
import contextlib
import logging
import re
import sqlite3
import sys
from pathlib import Path

%pip install matplotlib seaborn scikit-learn numpy pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")


def find_project_root(start=None) -> Path:
    """Walk upwards until a folder containing data/raw is found (works from notebooks/ or repo root)."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root (a folder containing data/raw).")


ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
for folder in (PROCESSED_DIR, FIG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DB_PATH = RAW_DIR / "PerfectPetClub.db"
CRM_PATH = RAW_DIR / "PerfectPetClub_CRM_report.csv"
LOG_PATH = RAW_DIR / "PerfectPetClub_server_log.csv"
SURVEY_PATH = RAW_DIR / "PerfectPetClub_survey.csv"
OUTPUT_PARQUET = PROCESSED_DIR / "processed_churn.parquet"

# Logging goes to stdout so it is visible in the exported HTML.
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True,
                    format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
logging.getLogger("matplotlib").setLevel(logging.WARNING)   # keep third-party chatter out of our log
log = logging.getLogger("etl")
log.info("Project root: %s", ROOT)

In [ ]:
class DataQualityError(Exception):
    """Raised when a critical data-quality check fails."""


class DataQualityReport:
    """Collects the outcome of every data-quality check made during the Transform stage."""

    def __init__(self):
        self.records = []

    def check(self, table: str, name: str, passed: bool, detail: str = "", critical: bool = False):
        status = "PASS" if passed else ("FAIL" if critical else "WARN")
        self.records.append({"table": table, "check": name, "status": status, "detail": detail})
        if passed:
            log.debug("DQ PASS | %s | %s", table, name)
        elif critical:
            log.error("DQ FAIL | %s | %s | %s", table, name, detail)
            raise DataQualityError(f"{table}: {name} - {detail}")
        else:
            log.warning("DQ WARN | %s | %s | %s", table, name, detail)

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.records)


def require_columns(df: pd.DataFrame, columns, table: str, dq: DataQualityReport):
    """Schema validation: every expected column must exist."""
    missing = [c for c in columns if c not in df.columns]
    dq.check(table, "schema: expected columns present", not missing, f"missing={missing}", critical=True)


dq = DataQualityReport()  # one report shared by the whole run

## 2. EXTRACT
Four sources: the SQLite sales database and three CSV files. Each extract function validates that the file exists, logs the row count, and raises a clear error otherwise.

> **Survey source note:** the brief says the survey is hosted on the university S3 store (VPN only). A copy is already in `data/raw/`, so the pipeline reads that. `download_survey_from_s3()` below shows how to refresh it; credentials are read from **environment variables** (never hard-coded in a repository).

In [ ]:
def extract_sqlite_tables(db_path: Path) -> dict:
    """Read every user table of the SQLite database into a dict of DataFrames."""
    if not db_path.exists():
        raise FileNotFoundError(f"Database not found: {db_path}")
    with contextlib.closing(sqlite3.connect(db_path)) as conn:
        names = [r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
        tables = {name: pd.read_sql_query(f"SELECT * FROM {name}", conn) for name in names}
    for name, frame in tables.items():
        log.info("Extracted table %-17s rows=%6d cols=%d", name, *frame.shape)
    return tables


def extract_csv(path: Path, label: str) -> pd.DataFrame:
    """Read one CSV file with basic error handling."""
    if not path.exists():
        raise FileNotFoundError(f"{label} file not found: {path}")
    frame = pd.read_csv(path)
    log.info("Extracted %-17s rows=%6d cols=%d", label, *frame.shape)
    return frame


def download_survey_from_s3(dest: Path = SURVEY_PATH):
    """OPTIONAL (UH VPN only). Credentials come from env vars S3_ENDPOINT, S3_ACCESS_KEY, S3_SECRET_KEY, S3_BUCKET."""
    import os
    import boto3  # imported lazily: only needed for this optional step
    s3 = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"],
                      aws_access_key_id=os.environ["S3_ACCESS_KEY"],
                      aws_secret_access_key=os.environ["S3_SECRET_KEY"])
    s3.download_file(os.environ["S3_BUCKET"], "PerfectPetClub_survey.csv", str(dest))
    log.info("Survey downloaded to %s", dest)


def extract() -> dict:
    """EXTRACT stage: returns every raw source in a single dictionary."""
    log.info("=== EXTRACT ===")
    raw = extract_sqlite_tables(DB_PATH)
    raw["crm"] = extract_csv(CRM_PATH, "CRM report")
    raw["server_log"] = extract_csv(LOG_PATH, "Server log")
    raw["survey"] = extract_csv(SURVEY_PATH, "Survey")
    return raw